# Timescale and trajectory curvature simulation

Illustrative 2D simulation comparing slow- and fast-varying signals with and without additive noise. Trajectory curvature is computed as the mean angle between successive displacement vectors. This notebook supports Fig. 1c-d.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'retina'
from scipy.stats import norm

# Set constants
nSteps = 20
timeVec = np.linspace(0, 1.5 * np.pi, nSteps)
ramp = norm.cdf(timeVec, loc=np.pi, scale=3 * np.pi / 4)

slowSignal1 = (1 - ramp) * np.sin(timeVec)
slowSignal2 = (1 - ramp) * np.cos(timeVec)

fastSignal1 = (1 - ramp) * np.sin(2 * timeVec)
fastSignal2 = (1 - ramp) * np.cos(2 * timeVec + np.pi / 4)

# Reproducible additive noise
np.random.seed(0)
noiseSeries1 = 0.1 * np.random.randn(len(timeVec))
noiseSeries2 = 0.1 * np.random.randn(len(timeVec))

In [ ]:
# Construct trajectories
slowSignalTraj = np.vstack([slowSignal1, slowSignal2])
slowNoisyTraj = np.vstack([slowSignal1 + noiseSeries1, slowSignal2 + noiseSeries2])
fastSignalTraj = np.vstack([fastSignal1, fastSignal2])
fastNoisyTraj = np.vstack([fastSignal1 + noiseSeries1, fastSignal2 + noiseSeries2])

trajectories = [slowSignalTraj, slowNoisyTraj, fastSignalTraj, fastNoisyTraj]
trajCurv = np.zeros(4)

# Compute trajectory curvature
for iT, traj in enumerate(trajectories):
    diffMat = np.diff(traj, axis=1)
    vectorLength = np.sqrt(np.sum(diffMat**2, axis=0))
    diffMatNl = diffMat / vectorLength

    cosAngles = np.sum(diffMatNl[:, :-1] * diffMatNl[:, 1:], axis=0)
    cosAngles = np.clip(cosAngles, -1, 1)
    trajCurv[iT] = np.mean(np.degrees(np.arccos(cosAngles)))

print(f"Slow signal curvature: {trajCurv[0]:.2f}°")
print(f"Slow noisy curvature: {trajCurv[1]:.2f}°")
print(f"Fast signal curvature: {trajCurv[2]:.2f}°")
print(f"Fast noisy curvature: {trajCurv[3]:.2f}°")

In [ ]:
# Plot signals and trajectories
fig, axes = plt.subplots(2, 2, figsize=(8, 8))

axes[0, 0].plot(timeVec, slowSignal1, label="Signal 1", color="black")
axes[0, 0].plot(timeVec, slowSignal1 + noiseSeries1, label="Signal 1 + noise", color="orange")
axes[0, 0].plot(timeVec, slowSignal2, label="Signal 2", color="black")
axes[0, 0].plot(timeVec, slowSignal2 + noiseSeries2, label="Signal 2 + noise", color="orange")
axes[0, 0].set_title("Slow signals")
axes[0, 0].set_box_aspect(1)

axes[0, 1].plot(slowSignal1, slowSignal2, label="Noiseless", color="black")
axes[0, 1].plot(slowSignal1 + noiseSeries1, slowSignal2 + noiseSeries2, label="Noisy", color="orange")
axes[0, 1].set_title(f"Slow trajectory: {trajCurv[0]:.1f}°, {trajCurv[1]:.1f}°")
axes[0, 1].set_box_aspect(1)

axes[1, 0].plot(timeVec, fastSignal1, label="Signal 1", color="black")
axes[1, 0].plot(timeVec, fastSignal1 + noiseSeries1, label="Signal 1 + noise", color="orange")
axes[1, 0].plot(timeVec, fastSignal2, label="Signal 2", color="black")
axes[1, 0].plot(timeVec, fastSignal2 + noiseSeries2, label="Signal 2 + noise", color="orange")
axes[1, 0].set_title("Fast signals")
axes[1, 0].set_box_aspect(1)

axes[1, 1].plot(fastSignal1, fastSignal2, label="Noiseless", color="black")
axes[1, 1].plot(fastSignal1 + noiseSeries1, fastSignal2 + noiseSeries2, label="Noisy", color="orange")
axes[1, 1].set_title(f"Fast trajectory: {trajCurv[2]:.1f}°, {trajCurv[3]:.1f}°")
axes[1, 1].set_box_aspect(1)

for ax in axes.flat:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

axes[0, 0].legend(frameon=False, fontsize=8)
axes[0, 1].legend(frameon=False, fontsize=8)
axes[1, 0].legend(frameon=False, fontsize=8)
axes[1, 1].legend(frameon=False, fontsize=8)

plt.tight_layout()
plt.show()